In [1]:
"""
Script to download and prepare data from Kaggle
"""
import os
import sys
# Add project root to path

import pandas as pd
import numpy as np
import kagglehub
from sklearn.preprocessing import LabelEncoder
"""
Configuration for Multi-Modal VAE training
"""
import torch


class Config:
    """Training and model configuration"""
    
    # Model architecture
    INPUT_DIM_A = None  # RNA expression dimension (computed from unique mapped genes)
    INPUT_DIM_B = None  # DNA methylation dimension (computed from unique mapped probes)
    LATENT_DIM = 20    # Latent space dimension 
    
    # Training parameters
    BATCH_SIZE = 32
    NUM_EPOCHS = 200
    LEARNING_RATE = 5e-4
    WEIGHT_DECAY = 1e-5
    
    # Loss parameters
    BETA_START = 1e-3  # KL divergence weight
    BETA_WARMUP_EPOCHS = 50  # Number of epochs for beta warmup
    GAMMA = 1.0  # Classification loss weight
    
    # Early stopping
    PATIENCE = 15
    
    # Optimizer
    LR_SCHEDULER_FACTOR = 0.5
    LR_SCHEDULER_PATIENCE = 5
    
    # Paths
    CHECKPOINT_DIR = 'checkpoints'
    BEST_MODEL_NAME = 'best_multivae.pt'
    
    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    # Data split
    TRAIN_TEST_SPLIT = 0.2
    RANDOM_SEED = 42


def download_datasets():
    """Download datasets from Kaggle"""
    print("Downloading RNA and mutations dataset...")
    rna_path = kagglehub.dataset_download('martininf1n1ty/plot-new-dataset')
    print(f"RNA dataset downloaded to: {rna_path}")
    
    print("\nDownloading DNA methylation dataset...")
    dna_path = kagglehub.dataset_download('martininf1n1ty/plot-new-dataset')
    print(f"DNA methylation dataset downloaded to: {dna_path}")
    
    return rna_path, dna_path

def prepare_rna_data(rna_path):
    """Prepare RNA expression data using dominant vector length (memory-safe)"""
    print("\nPreparing RNA expression data...")
    df_expressions = pd.read_parquet(
        f'{rna_path}/expression_new.parquet',
        columns=['case_barcode', 'gene_name', 'tpm_unstranded', 'primary_site']
    )

    def _is_listlike(x):
        return isinstance(x, (list, tuple, np.ndarray, pd.Series))

    has_nested_tpm = df_expressions['tpm_unstranded'].apply(_is_listlike).any()

    if has_nested_tpm:
        # Memory-safe path: no explode. Keep samples with the most common vector length.
        df_vectors = df_expressions.copy()
        df_vectors['tpm_unstranded'] = df_vectors['tpm_unstranded'].apply(
            lambda x: list(x) if _is_listlike(x) else [x]
        )
        df_vectors['tpm_len'] = df_vectors['tpm_unstranded'].apply(len)

        len_counts = df_vectors['tpm_len'].value_counts()
        target_len = int(len_counts.index[0])
        target_count = int(len_counts.iloc[0])

        print(f"Most frequent RNA vector length: {target_len} (samples: {target_count})")

        df_vectors = df_vectors[df_vectors['tpm_len'] == target_len].copy()
        df_vectors['tpm_unstranded'] = df_vectors['tpm_unstranded'].apply(
            lambda vals: [float(v) if pd.notna(v) else 0.0 for v in vals]
        )

        grouped_expressions_df = (
            df_vectors
            .sort_values(['case_barcode'])
            .groupby('case_barcode', as_index=False, observed=True)
            .agg({'tpm_unstranded': 'first', 'primary_site': 'first'})
        )

        Config.INPUT_DIM_A = target_len
        grouped_expressions_df = grouped_expressions_df[
            grouped_expressions_df['tpm_unstranded'].apply(len) == Config.INPUT_DIM_A
        ].reset_index(drop=True)

        print(f"Computed RNA INPUT_DIM_A (dominant length): {Config.INPUT_DIM_A}")
        print(f"RNA data shape: {grouped_expressions_df.shape}")
        return grouped_expressions_df

    # Fallback for long/tabular format: aggregate per patient then pick dominant length.
    df_expressions['gene_name'] = df_expressions['gene_name'].astype(str)
    df_expressions['tpm_unstranded'] = pd.to_numeric(df_expressions['tpm_unstranded'], errors='coerce').fillna(0.0)

    df_expressions_agg = (
        df_expressions
        .groupby(['case_barcode', 'gene_name'], as_index=False, observed=True)
        .agg({'tpm_unstranded': 'mean', 'primary_site': 'first'})
        .sort_values(['case_barcode', 'gene_name'])
    )

    grouped_expressions_df = (
        df_expressions_agg
        .groupby('case_barcode', as_index=False, observed=True)
        .agg({'tpm_unstranded': list, 'primary_site': 'first'})
    )

    grouped_expressions_df['tpm_len'] = grouped_expressions_df['tpm_unstranded'].apply(len)
    len_counts = grouped_expressions_df['tpm_len'].value_counts()
    target_len = int(len_counts.index[0])
    target_count = int(len_counts.iloc[0])

    Config.INPUT_DIM_A = target_len
    print(f"Most frequent RNA vector length: {target_len} (samples: {target_count})")

    grouped_expressions_df = grouped_expressions_df[
        grouped_expressions_df['tpm_len'] == Config.INPUT_DIM_A
    ].drop(columns=['tpm_len']).reset_index(drop=True)

    print(f"Computed RNA INPUT_DIM_A (dominant length): {Config.INPUT_DIM_A}")
    print(f"RNA data shape: {grouped_expressions_df.shape}")
    return grouped_expressions_df




def prepare_dna_methylation_data(dna_path):
    """Prepare DNA methylation data with memory-efficient common-feature mapping"""
    print("\nPreparing DNA methylation data...")
    df = pd.read_parquet(
        f'/kaggle/input/plot-new-dataset/part-00000-d32c9147-df07-4faa-821d-7cc24149ac16-c000.snappy.parquet',
        columns=['case_barcode', 'probe_id_id', 'beta_value']
    )

    df['probe_id_id'] = df['probe_id_id'].astype(str)
    df['beta_value'] = pd.to_numeric(df['beta_value'], errors='coerce')

    # Enforce unique (patient, probe) mapping and aggregate duplicates if present
    df_agg = (
        df
        .groupby(['case_barcode', 'probe_id_id'], as_index=False, observed=True)
        .agg({'beta_value': 'mean'})
    )

    n_cases = df_agg['case_barcode'].nunique()
    probe_case_counts = df_agg.groupby('probe_id_id', observed=True)['case_barcode'].nunique()
    common_probes = sorted(probe_case_counts[probe_case_counts == n_cases].index.tolist())

    Config.INPUT_DIM_B = len(common_probes)
    print(f"DNA patients: {n_cases}")
    print(f"Computed DNA INPUT_DIM_B (probes present in all patients): {Config.INPUT_DIM_B}")

    dna_common = df_agg[df_agg['probe_id_id'].isin(common_probes)].copy()
    dna_common = dna_common.sort_values(['case_barcode', 'probe_id_id'])

    grouped_df = (
        dna_common
        .groupby('case_barcode', as_index=False, observed=True)
        .agg({'beta_value': list})
    )

    grouped_df = grouped_df[
        grouped_df['beta_value'].apply(len) == Config.INPUT_DIM_B
    ].reset_index(drop=True)

    print(f"DNA methylation data shape: {grouped_df.shape}")
    return grouped_df


def merge_and_normalize_data(rna_df, dna_df, top_n_sites=24, output_dir='data'):
    """Merge all datasets and normalize"""
    print("\nMerging datasets...")
    os.makedirs(output_dir, exist_ok=True)
    
    # Merge RNA expression with DNA methylation using outer join to capture unmatched records
    merged_df = pd.merge(rna_df, dna_df, on='case_barcode', how='outer', indicator=True)
    
    # Identify and save unmatched records
    print("\nIdentifying unmatched records...")
    
    # RNA only (no matching DNA) - right side has NaN
    rna_only = merged_df[merged_df['_merge'] == 'left_only'].copy()
    if len(rna_only) > 0:
        print(f"Found {len(rna_only)} RNA samples without matching DNA methylation data")
        rna_only = rna_only[['case_barcode', 'tpm_unstranded', 'primary_site']]
        rna_only_path = os.path.join(output_dir, 'rna_only_unmatched.pkl')
        rna_only.to_pickle(rna_only_path)
        print(f"  Saved to: {rna_only_path}")
    else:
        print("No RNA-only samples found")
    
    # DNA only (no matching RNA) - left side has NaN
    dna_only = merged_df[merged_df['_merge'] == 'right_only'].copy()
    if len(dna_only) > 0:
        print(f"Found {len(dna_only)} DNA methylation samples without matching RNA expression data")
        dna_only = dna_only[['case_barcode', 'beta_value']]
        dna_only.to_pickle('data/dna_only_unmatched.pkl')
        print(f"  Saved to: data/dna_only_unmatched.pkl")
    else:
        print("No DNA-only samples found")
    
    # Keep only successfully merged records
    merged_df = merged_df[merged_df['_merge'] == 'both'].copy()
    merged_df = merged_df.drop(columns=['_merge'])
    
    print(f"\nMerged data shape before filtering: {merged_df.shape}")
    
    # Filter to keep only top N most common primary sites
    print(f"\nFiltering to keep only top {top_n_sites} most common primary sites...")
    site_counts = merged_df['primary_site'].value_counts()
    print(f"Total number of unique primary sites: {len(site_counts)}")
    
    top_sites = site_counts.head(top_n_sites).index.tolist()
    print(f"\nTop {top_n_sites} primary sites:")
    for i, (site, count) in enumerate(site_counts.head(top_n_sites).items(), 1):
        print(f"  {i}. {site}: {count} samples")
    
    # Filter dataframe to keep only top sites
    merged_df = merged_df[merged_df['primary_site'].isin(top_sites)].reset_index(drop=True)
    print(f"\nMerged data shape after filtering: {merged_df.shape}")
    
    # Normalize tpm_unstranded data
    print("\nNormalizing RNA expression data...")
    merged_df["tpm_unstranded"] = merged_df["tpm_unstranded"].apply(
        lambda x: np.log1p(np.array(x))
    )
    
    # Encode primary site labels
    print("\nEncoding primary site labels...")
    label_encoder = LabelEncoder()
    merged_df['primary_site_encoded'] = label_encoder.fit_transform(merged_df['primary_site'])
    
    print(f"\nPrimary site encoding (all {len(label_encoder.classes_)} classes):")
    for cls, code in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
        count = (merged_df['primary_site'] == cls).sum()
        print(f"  {code}: {cls} ({count} samples)")
    
    return merged_df, label_encoder


def main():
    """Main data preparation pipeline"""
    # Download datasets
    rna_path, dna_path = download_datasets()
    
    # Prepare individual datasets
    rna_df = prepare_rna_data(rna_path)
    dna_df = prepare_dna_methylation_data(dna_path)
    
    # Merge and normalize
    merged_df, label_encoder = merge_and_normalize_data(rna_df, dna_df)
    
    # Save processed data
    print("\nSaving processed data...")
    os.makedirs('data', exist_ok=True)
    merged_df.to_pickle('data/processed_data.pkl')
    
    # Save label encoder
    import pickle
    with open('data/label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)
    
    print("\nData preparation complete!")
    print(f"Processed data saved to: data/processed_data.pkl")
    print(f"Label encoder saved to: data/label_encoder.pkl")
    print(f"\nAdditional files (if any unmatched records):")
    print(f"  - data/rna_only_unmatched.pkl (RNA samples without DNA)")
    print(f"  - data/dna_only_unmatched.pkl (DNA samples without RNA)")


if __name__ == "__main__":
    main()



RNA dataset downloaded to: /kaggle/input/plot-new-dataset

DNA methylation dataset downloaded to: /kaggle/input/plot-new-dataset

Preparing RNA expression data...
Most frequent RNA vector length: 6800 (samples: 16456)
Computed RNA INPUT_DIM_A (dominant length): 6800
RNA data shape: (16456, 3)

Preparing DNA methylation data...
DNA patients: 10979
Computed DNA INPUT_DIM_B (probes present in all patients): 5133
DNA methylation data shape: (10979, 2)

Merging datasets...

Identifying unmatched records...
Found 6943 RNA samples without matching DNA methylation data
  Saved to: data/rna_only_unmatched.pkl
Found 1466 DNA methylation samples without matching RNA expression data
  Saved to: data/dna_only_unmatched.pkl

Merged data shape before filtering: (9513, 4)

Filtering to keep only top 24 most common primary sites...
Total number of unique primary sites: 57

Top 24 primary sites:
  1. Breast: 971 samples
  2. Bronchus and lung: 898 samples
  3. Kidney: 753 samples
  4. Brain: 680 samples

In [2]:
!git clone https://github.com/marcin119a/vae-los-angeles.git

Cloning into 'vae-los-angeles'...
remote: Enumerating objects: 256, done.
remote: Counting objects: 100% (256/256), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 256 (delta 134), reused 193 (delta 78), pack-reused 0 (from 0)
Receiving objects: 100% (256/256), 120.18 KiB | 6.68 MiB/s, done.
Resolving deltas: 100% (134/134), done.


In [3]:
%cd vae-los-angeles

/kaggle/working/vae-los-angeles


In [4]:
%mv /kaggle/working/data data

In [6]:
INPUT_DIM_A = 6800
INPUT_DIM_B = 5133

In [7]:
!git pull origin main

From https://github.com/marcin119a/vae-los-angeles
 * branch            main       -> FETCH_HEAD
Already up to date.


In [8]:
!INPUT_DIM_A=6800 INPUT_DIM_B=5133 LATENT_DIM=20 python3 train.py

Starting training run: 20260211_103550
Loading processed data...
Data shape: (8744, 5)
Number of primary sites: 24

Splitting data into train/validation sets...
Train set size: 6995
Validation set size: 1749

Computing class weights for balanced classification loss...
Class distribution in training set:
  Total classes in training set: 24 out of 24
  Class 0: 178 samples, weight: 1.6374
  Class 1: 291 samples, weight: 1.0016
  Class 2: 540 samples, weight: 0.5397
  Class 3: 793 samples, weight: 0.3675
  Class 4: 702 samples, weight: 0.4152
  ... and 19 more classes

Initializing model on cuda...

Starting training for 200 epochs...
Early stopping patience: 15
Epoch [1/200] | Train Loss: 438448.96 | Val Loss: 235596.60 | β=0.00000
✓ Best model saved (val_loss: 235596.60)
Epoch [2/200] | Train Loss: 210303.54 | Val Loss: 189506.01 | β=0.00002
✓ Best model saved (val_loss: 189506.01)
Epoch [3/200] | Train Loss: 184947.22 | Val Loss: 171659.92 | β=0.00004
✓ Best model saved (val_loss: 1716

In [9]:
!git pull origin main

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 824 bytes | 824.00 KiB/s, done.
From https://github.com/marcin119a/vae-los-angeles
 * branch            main       -> FETCH_HEAD
   37c7599..0ef75f8  main       -> origin/main
Updating 37c7599..0ef75f8
Fast-forward
 train_dna2rna.py | 11 ++++++++++-
 train_rna2dna.py | 11 ++++++++++-
 2 files changed, 20 insertions(+), 2 deletions(-)


In [10]:
!INPUT_DIM_A=6800 INPUT_DIM_B=5133 LATENT_DIM=20 python3 train_dna2rna.py

Starting DNA2RNAVAE training run: 20260211_104229
Loading processed data...
Data shape: (8744, 5)
Number of primary sites: 24

Splitting data into train/validation sets...
Train set size: 6995
Validation set size: 1749

Initializing DNA2RNAVAE model on cuda...

Starting training for 200 epochs...
Early stopping patience: 15
Epoch [1/200] | Train Loss: 348467.98 | Val Loss: 154820.94 | β=0.00000
✓ Best model saved (val_loss: 154820.94)
Epoch [2/200] | Train Loss: 140593.11 | Val Loss: 124301.97 | β=0.00002
✓ Best model saved (val_loss: 124301.97)
Epoch [3/200] | Train Loss: 116257.82 | Val Loss: 106950.21 | β=0.00004
✓ Best model saved (val_loss: 106950.21)
Epoch [4/200] | Train Loss: 101037.73 | Val Loss: 97783.33 | β=0.00006
✓ Best model saved (val_loss: 97783.33)
Epoch [5/200] | Train Loss: 93784.93 | Val Loss: 90466.50 | β=0.00008
✓ Best model saved (val_loss: 90466.50)
Epoch [6/200] | Train Loss: 88690.64 | Val Loss: 86646.13 | β=0.00010
✓ Best model saved (val_loss: 86646.13)
Epoc

In [11]:
!INPUT_DIM_A=6800 INPUT_DIM_B=5133 LATENT_DIM=20 python3 train_rna2dna.py

Starting RNA2DNAVAE training run: 20260211_104521
Loading processed data...
Data shape: (8744, 5)
Number of primary sites: 24

Splitting data into train/validation sets...
Train set size: 6995
Validation set size: 1749

Initializing RNA2DNAVAE model on cuda...

Starting training for 200 epochs...
Early stopping patience: 15
Epoch [1/200] | Train Loss: 85885.18 | Val Loss: 81845.55 | β=0.00000
✓ Best model saved (val_loss: 81845.55)
Epoch [2/200] | Train Loss: 81132.60 | Val Loss: 80133.71 | β=0.00002
✓ Best model saved (val_loss: 80133.71)
Epoch [3/200] | Train Loss: 79729.90 | Val Loss: 79979.78 | β=0.00004
✓ Best model saved (val_loss: 79979.78)
Epoch [4/200] | Train Loss: 79166.06 | Val Loss: 78634.12 | β=0.00006
✓ Best model saved (val_loss: 78634.12)
Epoch [5/200] | Train Loss: 78822.53 | Val Loss: 78439.31 | β=0.00008
✓ Best model saved (val_loss: 78439.31)
Epoch [6/200] | Train Loss: 78547.46 | Val Loss: 78304.96 | β=0.00010
✓ Best model saved (val_loss: 78304.96)
Epoch [7/200] 

In [12]:
!git pull origin main

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 574 bytes | 574.00 KiB/s, done.
From https://github.com/marcin119a/vae-los-angeles
 * branch            main       -> FETCH_HEAD
   0ef75f8..df69408  main       -> origin/main
Updating 0ef75f8..df69408
Fast-forward
 compare_directional_imputation.py | 10 ++++++++++
 1 file changed, 10 insertions(+)


In [13]:
!INPUT_DIM_A=6800 INPUT_DIM_B=5133 LATENT_DIM=20 python3 compare_directional_imputation.py

Loading processed data...
Loading RNA2DNAVAE from run: 20260211_104521
Loading DNA2RNAVAE from run: 20260211_104229

Computing baseline imputation predictions...
Computing mean imputation predictions...
Computing k-NN (k=5) predictions...

Generating predictions from RNA2DNA...

DNA Prediction Results:
  RNA2DNAVAE - MAE: 0.0746, MSE: 0.0131, R2: 0.8755, Pearson r: 0.9371
  Mean Imputation - MAE: 0.1341, MSE: 0.0323, R2: 0.6928, Pearson r: 0.8508
  k-NN Imputation - MAE: 0.0858, MSE: 0.0171, R2: 0.8369, Pearson r: 0.9203

Generating predictions from DNA2RNA...

RNA Prediction Results:
  DNA2RNAVAE - MAE: 0.3615, MSE: 0.3042, R2: 0.8894, Pearson r: 0.9467
  Mean Imputation - MAE: 0.5907, MSE: 0.7809, R2: 0.7160, Pearson r: 0.8520
  k-NN Imputation - MAE: 0.3935, MSE: 0.3738, R2: 0.8641, Pearson r: 0.9348

DIRECTIONAL VAE IMPUTATION COMPARISON RESULTS
       Modality           Model      MAE      MSE     RMSE       R2  CosineSimilarity  PearsonMean  PearsonStd
DNA methylation      RNA2DN

In [15]:
!INPUT_DIM_A=6800 INPUT_DIM_B=5133 LATENT_DIM=20 python3 reconstruct_unmatched.py

UNMATCHED DATA RECONSTRUCTION
Run timestamp: 20260211_105620

Loading label encoder...
✓ Label encoder loaded (24 classes)

Loading trained models...
Loading RNA2DNAVAE from run: 20260211_104521
✓ RNA2DNAVAE model loaded successfully
Loading DNA2RNAVAE from run: 20260211_104229
✓ DNA2RNAVAE model loaded successfully

Loading RNA-only samples from: data/rna_only_unmatched.pkl
  Filtered out 302 samples with unknown primary_site

RECONSTRUCTING DNA FROM RNA-ONLY SAMPLES
Number of RNA-only samples: 6641
✓ Reconstructed DNA shape: (6641, 5133)
✓ Saved reconstructed data to: data/rna_with_reconstructed_dna_20260211_105620.pkl

Loading DNA-only samples from: data/dna_only_unmatched.pkl

RECONSTRUCTING RNA FROM DNA-ONLY SAMPLES
Number of DNA-only samples: 1466

Note: DNA-only samples don't have primary_site information.
Trying reconstruction without site information (site=None)...
✓ Reconstructed RNA shape: (1466, 6800)
✓ Saved reconstructed data to: data/dna_with_reconstructed_rna_20260211_1

In [16]:
!python cluster_reconstructed.py

DIMENSIONALITY REDUCTION VISUALIZATION OF RECONSTRUCTED DATA
Run timestamp: 20260211_110145

Loading label encoder...
✓ Label encoder loaded (24 classes)
LOADING RECONSTRUCTED DATA

Loading RNA with reconstructed DNA from: data/rna_with_reconstructed_dna_20260211_105620.pkl
✓ Loaded 6641 RNA-only samples
  Columns: ['case_barcode', 'tpm_unstranded', 'primary_site', 'reconstructed_beta_value', 'primary_site_encoded']

Loading DNA with reconstructed RNA from: data/dna_with_reconstructed_rna_20260211_105620.pkl
✓ Loaded 1466 DNA-only samples
  Columns: ['case_barcode', 'beta_value', 'reconstructed_tpm_unstranded']

ANALYZING RNA-ONLY SAMPLES (with reconstructed DNA)
Number of samples: 6641
Feature matrix shape: (6641, 11933)

Primary site distribution:
  Hematopoietic and reticuloendothelial systems: 5126
  Kidney: 313
  Brain: 253
  Breast: 240
  Pancreas: 131
  Bronchus and lung: 123
  Colon: 118
  Cervix uteri: 85
  Ovary: 73
  Adrenal gland: 61
  Prostate gland: 38
  Retroperitoneum a